# Câu 1 - Thuật toán A* Manhattan

Tìm đường thoát từ phòng trung tâm ra rìa lâu đài trên ma trận `n x n`. Hai ô chỉ nối với nhau nếu có chung cạnh. Ô có giá trị `1` là đi được, ô có giá trị `0` là không đi được.

File đầu vào dùng trong bài là `A_in.csv`, file kết quả của thuật toán A* là `A_out.csv`.

## Đề bài và dữ liệu đầu vào

Dòng đầu của `A_in.csv` là:

```text
10,5,5
```

Ý nghĩa:

- `n = 10`: lâu đài là ma trận `10 x 10`.
- Vị trí bắt đầu là `(5,5)`.
- Tọa độ dùng kiểu `0-based`, tức dòng/cột bắt đầu từ `0`.

Mục tiêu là tìm một đường đi từ `(5,5)` đến một ô bất kỳ nằm trên rìa ma trận.

## Định nghĩa khoảng cách Manhattan và công thức h(x)

Khoảng cách Manhattan là khoảng cách di chuyển trên lưới khi chỉ được đi theo các hướng ngang hoặc dọc, không được đi chéo.

Với hai ô:

```text
A = (x1, y1)
B = (x2, y2)
```

khoảng cách Manhattan giữa `A` và `B` là:

```text
d(A, B) = |x1 - x2| + |y1 - y2|
```

Trong bài lâu đài, mục tiêu không phải một ô cố định mà là bất kỳ ô nào nằm trên rìa ma trận. Với một ô hiện tại:

```text
x = (row, col)
```

rìa lâu đài gồm bốn cạnh:

- rìa trên: `row = 0`
- rìa trái: `col = 0`
- rìa dưới: `row = n - 1`
- rìa phải: `col = n - 1`

Khoảng cách Manhattan ngắn nhất từ ô `x` đến rìa gần nhất là:

```text
h(x) = min(
    |row - 0|,
    |col - 0|,
    |row - (n - 1)|,
    |col - (n - 1)|
)
```

Vì `row`, `col` đều nằm trong ma trận nên có thể viết gọn:

```text
h(x) = min(row, col, n - 1 - row, n - 1 - col)
```

Ví dụ với `n = 10`, ô bắt đầu `(5,5)`:

```text
Đến rìa trên:  |5 - 0| = 5
Đến rìa trái:  |5 - 0| = 5
Đến rìa dưới:  |5 - 9| = 4
Đến rìa phải:  |5 - 9| = 4

h(5,5) = min(5, 5, 4, 4) = 4
```

Nếu xét ô `(7,9)`:

```text
Đến rìa phải: |9 - 9| = 0
h(7,9) = 0
```

Vì `h(7,9) = 0`, ô `(7,9)` nằm trên rìa và là một cửa ra hợp lệ.

Trong code, công thức này được cài đặt bằng hàm:

```python
def heuristic_to_border(position, n):
    row, col = position
    distances = [
        abs(row - 0),
        abs(col - 0),
        abs(row - (n - 1)),
        abs(col - (n - 1)),
    ]
    return min(distances)
```

## Biểu diễn bài toán dưới dạng đồ thị

Bài toán lâu đài có thể xem như một bài toán tìm đường trên đồ thị:

- Mỗi ô trong ma trận là một đỉnh của đồ thị.
- Chỉ những ô có giá trị `1` mới được xem là đỉnh hợp lệ vì đó là ô có đường hầm và có thể đi được.
- Hai đỉnh có cạnh nối với nhau nếu hai ô tương ứng nằm kề cạnh theo một trong bốn hướng: trên, phải, dưới, trái.
- Ô bắt đầu là `(5,5)`.
- Tập đích không chỉ có một ô, mà là tất cả các ô đi được nằm trên rìa ma trận.

Vì vậy, thay vì phải tạo danh sách cạnh trước, chương trình sinh các cạnh ngay trong lúc chạy bằng hàm `get_neighbors()`. Khi thuật toán đang đứng tại một ô, hàm này kiểm tra bốn ô xung quanh và chỉ trả về những ô hợp lệ. Cách làm này gọn hơn vì đồ thị được suy ra trực tiếp từ ma trận.

Ví dụ, nếu đang ở ô `(5,5)`, chương trình xét bốn hướng:

```text
(4,5)  đi lên
(5,6)  đi sang phải
(6,5)  đi xuống
(5,4)  đi sang trái
```

Sau đó chỉ giữ lại những ô nằm trong ma trận và có giá trị `1`. Đây chính là bước mở rộng trạng thái trong thuật toán A*.

## b) Code chương trình A*

Dưới đây là chương trình A* hiện có trong file `cau1_astar.py`. Notebook chỉ sao chép nội dung để trình bày, không chỉnh sửa file code gốc.

In [ ]:
from pathlib import Path
import csv
import heapq

import matplotlib.pyplot as plt


def read_input(file_path):
    with open(file_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        first_line = next(reader)
        n = int(first_line[0])
        start = (int(first_line[1]), int(first_line[2]))
        maze = [[int(value) for value in row] for row in reader]

    return n, start, maze


def heuristic_to_border(position, n):
    row, col = position
    distances = [
        abs(row - 0),
        abs(col - 0),
        abs(row - (n - 1)),
        abs(col - (n - 1)),
    ]
    return min(distances)


def is_inside(position, n):
    row, col = position
    return 0 <= row < n and 0 <= col < n


def is_border(position, n):
    row, col = position
    return row == 0 or row == n - 1 or col == 0 or col == n - 1


def get_neighbors(position, n, maze):
    row, col = position
    directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    neighbors = []

    for d_row, d_col in directions:
        next_pos = (row + d_row, col + d_col)

        if is_inside(next_pos, n) and maze[next_pos[0]][next_pos[1]] == 1:
            neighbors.append(next_pos)

    return neighbors


def reconstruct_path(parent, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def astar_escape(n, start, maze):
    if not is_inside(start, n) or maze[start[0]][start[1]] != 1:
        return None

    open_set = []
    start_g = 0
    start_h = heuristic_to_border(start, n)
    heapq.heappush(open_set, (start_g + start_h, start_h, start_g, start))

    parent = {start: None}
    g_score = {start: 0}
    visited = set()

    while open_set:
        _, _, current_g, current = heapq.heappop(open_set)

        if current in visited:
            continue

        visited.add(current)

        if is_border(current, n):
            return reconstruct_path(parent, current)

        for neighbor in get_neighbors(current, n, maze):
            tentative_g = current_g + 1

            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                parent[neighbor] = current
                h = heuristic_to_border(neighbor, n)
                f = tentative_g + h
                heapq.heappush(open_set, (f, h, tentative_g, neighbor))

    return None


def write_output(file_path, path):
    with open(file_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)

        if path is None:
            writer.writerow([-1])
            return

        writer.writerow([len(path)])
        writer.writerows(path)


def save_path_chart(maze, path, output_file):
    n = len(maze)

    plt.figure(figsize=(6, 6))
    plt.imshow(maze, cmap="gray_r")
    plt.xticks(range(n))
    plt.yticks(range(n))
    plt.grid(color="lightgray", linewidth=0.5)

    if path is not None:
        rows = [position[0] for position in path]
        cols = [position[1] for position in path]
        plt.plot(cols, rows, color="red", linewidth=2.5, marker="o", markersize=5, label="Duong di A* Manhattan")
        plt.scatter(cols[0], rows[0], c="lime", s=140, edgecolors="black", label="Bat dau")
        plt.scatter(cols[-1], rows[-1], c="yellow", s=140, edgecolors="black", label="Cua ra")

    plt.title("Duong thoat tim duoc bang A* voi h Manhattan")
    plt.xlabel("Cot")
    plt.ylabel("Dong")
    plt.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(output_file, dpi=160)
    plt.close()


def main():
    current_dir = Path(__file__).resolve().parent
    input_file = current_dir.parent / "A_in.csv"
    output_file = current_dir / "A_out.csv"
    path_image = current_dir / "cau1_astar_path.png"

    n, start, maze = read_input(input_file)
    path = astar_escape(n, start, maze)
    write_output(output_file, path)
    save_path_chart(maze, path, path_image)

    if path is None:
        print("Khong tim thay duong thoat. A_out.csv ghi -1.")
    else:
        print(f"Tim thay duong thoat bang A*: {len(path)} o.")
        print(f"Da ghi ket qua vao: {output_file}")
        print(f"Da luu bieu do duong di: {path_image}")


if __name__ == "__main__":
    main()

## Giải thích tác dụng của từng hàm

- `read_input(file_path)`: đọc file CSV đầu vào, lấy kích thước ma trận `n`, vị trí bắt đầu `start` và ma trận lâu đài `maze`.
- `heuristic_to_border(position, n)`: tính khoảng cách Manhattan ngắn nhất từ ô hiện tại đến rìa gần nhất của ma trận. Đây là hàm `h(x)` dùng trong A*.
- `is_inside(position, n)`: kiểm tra một tọa độ có nằm trong phạm vi ma trận `n x n` hay không.
- `is_border(position, n)`: kiểm tra một ô có nằm ở rìa lâu đài hay không. Nếu có, ô đó là cửa ra hợp lệ.
- `get_neighbors(position, n, maze)`: sinh các ô kề cạnh hợp lệ theo bốn hướng trên, phải, dưới, trái. Chỉ những ô nằm trong ma trận và có giá trị `1` mới được đi tiếp.
- `reconstruct_path(parent, goal)`: truy vết đường đi từ ô cửa ra về ô bắt đầu dựa trên dictionary `parent`, sau đó đảo ngược lại để có đường đi đúng chiều.
- `astar_escape(n, start, maze)`: thực hiện thuật toán A*. Hàm dùng hàng đợi ưu tiên `open_set`, luôn lấy ô có `f = g + h` nhỏ nhất ra xét trước, cập nhật đường đi tốt hơn cho các ô kề và trả về đường thoát nếu tìm thấy rìa.
- `write_output(file_path, path)`: ghi kết quả ra file CSV. Nếu không tìm thấy đường thì ghi `-1`, nếu tìm thấy thì ghi số ô của đường đi và danh sách tọa độ.
- `save_path_chart(maze, path, output_file)`: vẽ ma trận lâu đài và đường đi A* tìm được, sau đó lưu thành ảnh PNG.
- `main()`: hàm điều phối chính, đọc input, chạy A*, ghi output, lưu biểu đồ và in thông báo kết quả.

## Giải thích thuật toán A*

A* là thuật toán tìm kiếm có thông tin. Trong bài này, A* vừa xét chi phí thật đã đi từ phòng trung tâm, vừa dùng heuristic Manhattan để ước lượng khoảng cách còn lại đến rìa lâu đài.

Mỗi trạng thái là một ô `(row, col)` trong ma trận. Mỗi lần đi sang ô kề cạnh hợp lệ thì chi phí tăng thêm `1`.

A* sử dụng công thức:

```text
f(x) = g(x) + h(x)
```

Trong đó:

- `g(x)`: số bước thật từ ô bắt đầu `(5,5)` đến ô `x`.
- `h(x)`: khoảng cách Manhattan ngắn nhất từ ô `x` đến rìa gần nhất.
- `f(x)`: tổng điểm ưu tiên. Ô có `f` nhỏ hơn được xét trước.

Quy trình thuật toán trong code:

1. Kiểm tra ô bắt đầu có hợp lệ không. Nếu `(5,5)` nằm ngoài ma trận hoặc có giá trị `0` thì trả về `None`.
2. Khởi tạo `open_set` là hàng đợi ưu tiên. Đưa ô bắt đầu vào với `g = 0`, `h = heuristic_to_border(start, n)` và `f = g + h`.
3. Khởi tạo `parent` để lưu đường đi, `g_score` để lưu chi phí tốt nhất đến từng ô, và `visited` để tránh xử lý lại ô đã xét chính thức.
4. Trong vòng lặp, lấy ô có `f` nhỏ nhất ra khỏi `open_set` bằng `heapq.heappop()`.
5. Nếu ô vừa lấy ra đã nằm trên rìa ma trận, thuật toán gọi `reconstruct_path()` để truy vết đường đi và kết thúc.
6. Nếu chưa tới rìa, gọi `get_neighbors()` để sinh các ô kề cạnh hợp lệ.
7. Với mỗi ô kề, tính `tentative_g = current_g + 1`. Nếu đây là đường đi đầu tiên đến ô đó hoặc tốt hơn chi phí cũ, cập nhật `g_score`, cập nhật `parent`, tính lại `h`, `f`, rồi đưa ô đó vào `open_set`.
8. Nếu `open_set` rỗng mà chưa gặp ô rìa, nghĩa là không tìm thấy đường thoát.

Ý nghĩa của vòng lặp chính:

```text
Lặp khi open_set còn phần tử:
    lấy ô có f nhỏ nhất
    nếu là ô rìa -> dựng đường đi và dừng
    nếu chưa -> mở rộng các ô kề hợp lệ
```

A* khác BFS ở chỗ không chỉ xét theo số bước đã đi. Nó kết hợp `g` và `h`, nên có xu hướng đi về phía rìa gần hơn nhưng vẫn kiểm soát chi phí thật đã đi. Với bài này, đường đi tìm được là `(5,5) -> (5,6) -> (5,7) -> (5,8) -> (6,8) -> (7,8) -> (7,9)`.

## Phân tích chi tiết luồng xử lý trong A*

Hàm chính của thuật toán là `astar_escape(n, start, maze)`.

Trước tiên, chương trình kiểm tra ô bắt đầu:

```python
if not is_inside(start, n) or maze[start[0]][start[1]] != 1:
    return None
```

Nếu ô bắt đầu nằm ngoài ma trận hoặc không đi được thì bài toán không có lời giải.

Sau đó A* khởi tạo các cấu trúc dữ liệu chính:

- `open_set`: hàng đợi ưu tiên, chứa các ô sẽ được xét.
- `parent`: lưu cha của mỗi ô để sau khi tìm thấy cửa ra có thể truy vết lại đường đi.
- `g_score`: lưu chi phí tốt nhất hiện biết từ ô bắt đầu đến từng ô.
- `visited`: lưu các ô đã được lấy ra xử lý chính thức, tránh xử lý lặp lại.

Với ô bắt đầu `(5,5)`:

```text
g = 0
h = 4
f = g + h = 4
```

Nên phần tử đầu tiên được đưa vào `open_set` là:

```text
(4, 4, 0, (5,5))
```

Ở mỗi vòng lặp, A* lấy ô có `f` nhỏ nhất ra khỏi `open_set`. Nếu ô đó nằm trên rìa thì thuật toán dừng và gọi `reconstruct_path()` để tạo đường đi. Nếu chưa tới rìa, thuật toán sinh các ô kề bằng `get_neighbors()`, tính chi phí mới `tentative_g = current_g + 1`, rồi cập nhật nếu đường đi mới tốt hơn đường đi đã biết trước đó.

Điểm khác biệt quan trọng của A* so với BFS là A* không chỉ xét theo số bước đã đi. Nó kết hợp cả:

- chi phí thật đã đi: `g`
- khoảng cách ước lượng còn lại: `h`

Nhờ đó thuật toán có xu hướng đi về phía rìa gần hơn, nhưng vẫn không bỏ qua chi phí thực tế đã đi.

## Bảng bước chạy chi tiết của A*

| Bước | Ô lấy ra | g | h | f | Thêm vào open_set | Ghi chú |
|---:|---|---:|---:|---:|---|---|
| 1 | `(5,5)` | 0 | 4 | 4 | `(4,5)` g=1 h=4 f=5; `(5,6)` g=1 h=3 f=4 | Tiếp tục |
| 2 | `(5,6)` | 1 | 3 | 4 | `(5,7)` g=2 h=2 f=4 | Tiếp tục |
| 3 | `(5,7)` | 2 | 2 | 4 | `(5,8)` g=3 h=1 f=4 | Tiếp tục |
| 4 | `(5,8)` | 3 | 1 | 4 | `(4,8)` g=4 h=1 f=5; `(6,8)` g=4 h=1 f=5 | Tiếp tục |
| 5 | `(4,8)` | 4 | 1 | 5 | `(3,8)` g=5 h=1 f=6 | Tiếp tục |
| 6 | `(6,8)` | 4 | 1 | 5 | `(7,8)` g=5 h=1 f=6 | Tiếp tục |
| 7 | `(4,5)` | 1 | 4 | 5 | `(3,5)` g=2 h=3 f=5 | Tiếp tục |
| 8 | `(3,5)` | 2 | 3 | 5 | `(3,4)` g=3 h=3 f=6 | Tiếp tục |
| 9 | `(3,8)` | 5 | 1 | 6 | `(2,8)` g=6 h=1 f=7 | Tiếp tục |
| 10 | `(7,8)` | 5 | 1 | 6 | `(7,9)` g=6 h=0 f=6 | Tiếp tục |
| 11 | `(7,9)` | 6 | 0 | 6 | Không thêm | Gặp cửa ra |

A* tìm thấy cửa ra tại ô `(7,9)`. Đường đi có `7` ô.

## Biểu đồ đường đi A*

![Đường thoát tìm được bằng A*](cau1_astar_path.png)

Ý nghĩa:

- Ô màu đen là ô đi được, tương ứng giá trị `1`.
- Ô màu trắng là tường/không đi được, tương ứng giá trị `0`.
- Đường màu đỏ là đường đi A*.
- Điểm màu xanh là phòng trung tâm.
- Điểm màu vàng là cửa ra.

## Giải thích ma trận và hình minh họa

Trong hình minh họa, ma trận được vẽ bằng `plt.imshow(maze, cmap=gray_r)`:

- Ô có giá trị `1` là ô đi được.
- Ô có giá trị `0` là tường hoặc không có đường hầm.
- Đường màu đỏ nối các tọa độ trong danh sách đường đi mà A* tìm được.
- Điểm đầu của đường đỏ là `(5,5)`, tức phòng trung tâm.
- Điểm cuối là `(7,9)`, nằm ở cột cuối cùng của ma trận `10 x 10`, nên đây là cửa ra ở rìa phải.

Đường đi trong `A_out.csv` là:

```text
(5,5) -> (5,6) -> (5,7) -> (5,8) -> (6,8) -> (7,8) -> (7,9)
```

Đường đi này hợp lệ vì mỗi cặp ô liên tiếp đều kề cạnh nhau, không đi chéo, và tất cả các ô trên đường đi đều có giá trị `1` trong ma trận đầu vào.

Số `7` ở dòng đầu của `A_out.csv` là số ô trong đường đi, không phải số cạnh. Nếu tính số bước di chuyển thì đường đi có `6` bước.

## c) Kết quả thực thi

Lệnh chạy chương trình:

```powershell
python 2025/De1/BFS_DFS_Astar_Manhattan_Cau1/01_AStar/cau1_astar.py
```

Kết quả trong `A_out.csv`:

```text
7
5,5
5,6
5,7
5,8
6,8
7,8
7,9
```

Kết luận: A* tìm được đường thoát hợp lệ gồm `7` ô từ `(5,5)` đến cửa ra `(7,9)`.